In [5]:
!python3 -m pip install requests bs4


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import requests
import json
import os
import re
import time
from bs4 import BeautifulSoup

In [7]:
DATA_DIR = "data/literature"
os.makedirs(DATA_DIR, exist_ok=True)

In [8]:
def build_query(gene, variant):
    return f'("{gene}" AND "{variant}") AND (mutation OR pathogenic OR aggregation OR misfolding OR "protein stability")'

In [9]:
def clean_text(text):
    if not text:
        return ""
    
    return BeautifulSoup(text, "html.parser").get_text(" ", strip=True)

In [10]:
def fetch_europepmc(query, max_papers=100):
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    
    papers = []
    seen = set()
    cursor = "*"
    
    while len(papers) < max_papers:
        params = {
            "query": query,
            "format": "json",
            "pageSize": 50,
            "resultType": "core",
            "cursorMark": cursor
        }
        
        # 🔁 Retry logic with backoff
        for attempt in range(3):
            try:
                time.sleep(1)  # rate limiting
                
                resp = requests.get(url, params=params, timeout=10)
                data = resp.json()
                break
            
            except Exception as e:
                print(f"⚠️ Retry {attempt+1} failed: {e}")
                time.sleep(2 ** attempt)
        else:
            print("❌ API failed after retries")
            return papers
        
        results = data.get("resultList", {}).get("result", [])
        
        if not results:
            break
        
        for item in results:
            # ✅ Fix: PMID fallback
            pmid = item.get("pmid") or item.get("id")
            abstract = clean_text(item.get("abstractText", ""))
            
            # ✅ Filtering (CRITICAL)
            if not pmid:
                continue
            if not abstract or len(abstract) < 100:
                continue
            
            # ✅ Deduplication
            if pmid in seen:
                continue
            seen.add(pmid)
            
            # ✅ Consistent schema
            papers.append({
                "pmid": pmid,
                "title": item.get("title", ""),
                "abstract": abstract
            })
            
            if len(papers) >= max_papers:
                break
        
        # ✅ Pagination
        cursor = data.get("nextCursorMark")
        if not cursor:
            break
    
    print(f"✅ Final valid papers: {len(papers)}")
    return papers

In [11]:
def save_corpus(gene, variant, papers):
    filename = f"{gene}_{variant}_corpus.json"
    path = os.path.join(DATA_DIR, filename)
    
    if not papers:
        print("⚠️ No literature found for this mutation")
    
    with open(path, "w") as f:
        json.dump(papers, f, indent=4, ensure_ascii=False)
    
    print(f"✅ Saved to {path}")

In [12]:
def fetch_literature(gene, variant):
    query = build_query(gene, variant)
    
    print(f"🔍 Query: {query}")
    
    papers = fetch_europepmc(query)
    
    save_corpus(gene, variant, papers)
    
    return papers

In [13]:
papers = fetch_literature("SNCA", "A53T")

papers[:3]

🔍 Query: ("SNCA" AND "A53T") AND (mutation OR pathogenic OR aggregation OR misfolding OR "protein stability")
✅ Final valid papers: 100
✅ Saved to data/literature/SNCA_A53T_corpus.json


[{'pmid': '41002401',
  'title': 'Modeling Synucleinopathy Using hESC-Derived Cerebral Organoids.',
  'abstract': 'Animal and cellular models harboring SNCA gene mutations have been instrumental in synucleinopathy, but faithful human brain models remain limited. Here, we report the development of a human cerebral organoid (CO) model of synucleinopathy carrying the Ala53Thr mutation in SNCA ( SNCA A53T ). Using a human embryonic stem cell (hESC) line overexpressing SNCA A53T (A53T hESC line), we generated COs (A53T COs) that recapitulate hallmark features of synucleinopathy. These A53T COs exhibited elevated α-synuclein (α-Syn) expression, the increased phosphorylation of α-Syn, and Lewy body-like aggregations. Notably, we also observed the increased expression of phosphorylated tau and neurofibrillary tangle-like silver deposits, although amyloid β expression and accumulation remained unchanged. To evaluate the utility of this model in drug screening, we treated A53T COs with synuclean

In [14]:
print("Total papers:", len(papers))

# Check all PMIDs valid
print("All PMIDs valid:", all(p['pmid'] is not None for p in papers))

# Check abstract quality
print("Min abstract length:", min(len(p['abstract']) for p in papers) if papers else 0)

Total papers: 100
All PMIDs valid: True
Min abstract length: 508


In [15]:
from google.colab import files
import json
import os

# Download the generated corpus file to your local machine
corpus_path = "/content/data/literature/SNCA_A53T_corpus.json"
if os.path.exists(corpus_path):
    print(f"📥 Downloading {corpus_path} to your local machine...")
    files.download(corpus_path)
    print("✅ Download complete! Check your Downloads folder.")
else:
    print("❌ File not found")

ModuleNotFoundError: No module named 'google'

In [ ]:
papers[7]

{'pmid': '41277874',
 'title': 'From Yeast to Therapeutics: Modeling Neurodegenerative Diseases in Saccharomyces cerevisiae.',
 'abstract': "Here, we review the use of Saccharomyces cerevisiae as a powerful model organism for studying cellular processes implicated in neurodegenerative disorders, including stress responses, proteostasis impairment, and vesicle trafficking defects. Over the last two decades, baker's yeast models have been developed for complex diseases such as Parkinson's, Alzheimer's, Huntington's, and Amyotrophic lateral sclerosis (ALS). Yeast cells expressing human proteins, such as amyloid-β, α-synuclein, huntingtin, and TDP-43, have become crucial tools for high-throughput drug screening aimed at counteracting disease progression. These yeast models have unveiled key components involved in the metabolism and toxicity of these proteins, enabling the identification of interacting partners and novel factors within each pathway. Importantly, these pathways were subseque

In [ ]:
papers[4]

{'pmid': '41703390',
 'title': "Pathological Protein Targets in Parkinson's Disease: Progress Towards the Development of Disease-Modifying Therapies.",
 'abstract': "Parkinson's disease (PD) is a common neurodegenerative movement disorder that currently has no disease-modifying therapies. Over the past two decades, there has been a substantial acceleration in the knowledge of how genetics underlies PD risk. This has given rise to pathological protein targets that can be therapeutically targeted. In particular, there is compelling evidence for developing therapeutic strategies targeting alpha-synuclein, leucine-rich repeat kinase 2 (LRRK2) and glucocerebrosidase (GCase). These proteins are implicated in lysosomal function and may contribute to the accumulation of the hallmark pathological forms of alpha-synuclein that define PD. This review highlights current progress on PD therapies targeting these proteins as they attempt to make their way through the clinical trial pipeline, with uni